In [ ]:
## Setup

import numpy as np
import pandas as pd


from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
accuracy_score, classification_report, confusion_matrix,
mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
ratio = 0.2

X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=ratio, random_state=42, stratify=y if y.nunique() < 20 else None
)

In [ ]:
# Preprocesado típico (numéricas + categóricas)
num_features = X.select_dtypes(include=["number"]).columns
cat_features = X.select_dtypes(exclude=["number"]).columns


numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()) ])


categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")) ])


preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_features),
        ("cat", categorical_pipe, cat_features)    ],
    remainder="drop"    )

## Regresión lineal

regresión (variable numérica) como baseline rápido; relaciones aproximadamente lineales; interpretabilidad.

In [ ]:
from sklearn.linear_model import LinearRegression


model = Pipeline(steps=[
("prep", preprocess),
("reg", LinearRegression())
])


model.fit(X_train, y_train)
pred = model.predict(X_test)


rmse = mean_squared_error(y_test, pred, squared=False)
r2 = r2_score(y_test, pred)
rmse, r2

## Regresión logística

clasificación (binaria o multiclase) como baseline robusto; interpretabilidad; rápido.

In [ ]:
from sklearn.linear_model import LogisticRegression


model = Pipeline(steps=[
("prep", preprocess),
("clf", LogisticRegression(max_iter=2000))
])


model.fit(X_train, y_train)
pred = model.predict(X_test)


print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

## Árbol de decisión

clasificación/regresión; reglas interpretables; captura no linealidades; puede sobreajustar si no se limita.

In [ ]:
from sklearn.tree import DecisionTreeClassifier


model = Pipeline(steps=[
("prep", preprocess),
("clf", DecisionTreeClassifier(
max_depth=6, # controla complejidad
random_state=42
))
])


model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

## Random Forest

Reduce overfitting vs un árbol; buena performance sin demasiado tuning.

In [ ]:
from sklearn.ensemble import RandomForestClassifier


model = Pipeline(steps=[
("prep", preprocess),
("clf", RandomForestClassifier(
n_estimators=300,
max_depth=None,
random_state=42,
n_jobs=-1
))
])


model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

## GBoost / XGBoost-like

tabular con alta precisión; suele rendir muy bien; sensible a hiperparámetros; útil cuando quieres performance.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier


model = Pipeline(steps=[
("prep", preprocess),
("clf", HistGradientBoostingClassifier(
max_depth=6,
learning_rate=0.05,
max_iter=300,
random_state=42
))
])


model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

## k-Nearest Neighbors

baseline simple; no asume forma funcional; sensible a escalado; peor en alta dimensionalidad.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier


model = Pipeline(steps=[
("prep", preprocess),
("clf", KNeighborsClassifier(n_neighbors=15))
])


model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

## Naive Bayes

baseline clásico en NLP

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB


X_train, X_test, y_train, y_test = train_test_split(texts, y, test_size=0.2, random_state=42, stratify=y)


model = Pipeline(steps=[
("tfidf", TfidfVectorizer(max_features=50_000, ngram_range=(1,2))),
("clf", MultinomialNB())
])


model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

## Clustering: K-Means

segmentación sin etiquetas; exploración; requiere escalado; elegir K (p. ej., elbow/silhouette)

In [ ]:
from sklearn.cluster import KMeans


# X_num: matriz solo numérica ya imputada/escalada
X_num = X[num_features].copy()
X_num = SimpleImputer(strategy="median").fit_transform(X_num)
X_num = StandardScaler().fit_transform(X_num)


kmeans = KMeans(n_clusters=4, random_state=42, n_init="auto")
labels = kmeans.fit_predict(X_num)
labels[:10]

## Reducción de dimensionalidad: PCA

visualización; compresión; eliminar colinealidad; acelerar modelos.

In [ ]:
from sklearn.decomposition import PCA


X_num = X[num_features].copy()
X_num = SimpleImputer(strategy="median").fit_transform(X_num)
X_num = StandardScaler().fit_transform(X_num)


pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_num)
X_2d.shape

## Validación rápida (cross-validation)

estimar rendimiento general y detectar overfitting (si train muy alto y CV cae).

In [ ]:
from sklearn.model_selection import StratifiedKFold


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
print(scores.mean(), scores.std())